# 33 · Additional analyses (GPU recommended; resumable; no training)
Run after notebook 31 has finished. Read-only for runs and data; writes only `results/p2_rev5/addl/`.
* **A** what the trained circuits compute (function class, entanglement), at initialization and after training;
* **B** noisy re-evaluation of the trained circuit models (no retraining);
* **C** training dynamics and seed variability of all 200 runs.

In [ ]:
# 0 · Settings
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, itertools, gc
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
RES      = f'{PROJECT}/results'
OUT      = f'{RES}/p2_rev5/addl'; os.makedirs(OUT, exist_ok=True)
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
ROOTS    = {'train': f'{RES}/p2_jit_runs_train', 'val': f'{RES}/p2_jit_runs_val', 'train4': f'{RES}/p2_rev4_runs_train', 'val4': f'{RES}/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
QSEG_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p_ in (QSEG_DIR, f'{PROJECT}/code'):
    if p_ not in sys.path: sys.path.insert(0, p_)
import tensorflow as tf, qseg2, qseg2_rev4; qseg2_rev4.patch()
from qseg.config import Config
from qseg.quantum_sim import circuit_expvals, NumpyOps, _ring_perms
from scipy.optimize import least_squares
cfg = Config(work_dir=OUT); B = cfg.batch_size; SHAPE = (cfg.image_size, cfg.image_size, cfg.in_channels)
print('qseg package:', QSEG_DIR, '| TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · Data, runs, circuit simulation (pure and noisy) with a self-test against the training simulator
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
MD = pd.read_csv(f'{DMID_DIR}/metadata.csv')
XD = np.stack([_png(f'{DMID_DIR}/images/{r}.png', False) for r in MD.roi_id]); YD = np.stack([_png(f'{DMID_DIR}/masks/{r}.png', True) for r in MD.roi_id])
def runs_of(root):
    return [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
ALL = [(k, d) for k, r in ROOTS.items() for d in runs_of(r)]
CIRC = [(k, d) for k, d in ALL if json.load(open(f'{d}/summary.json'))['model'] in ('g_q', 'p_q')]
print('runs:', len(ALL), '| circuit runs:', len(CIRC), '| CBIS', XC.shape, '| DMID', XD.shape)

N = 4; DIM = 2 ** N
I2 = np.eye(2); PX = np.array([[0, 1], [1, 0]], complex); PY = np.array([[0, -1j], [1j, 0]]); PZ = np.diag([1.0, -1.0]).astype(complex)
def op_on(o, j):                       # wire 0 = most significant bit (as in qseg.quantum_sim)
    m = np.array([[1.0 + 0j]])
    for i in range(N): m = np.kron(m, o if i == j else I2)
    return m
PAULI = [[op_on(o, j) for o in (PX, PY, PZ)] for j in range(N)]
ZOPS = [op_on(PZ, j).real.diagonal() for j in range(N)]
RING = []
for p in _ring_perms(N):                # applied in this order, as flat[:, p]
    P = np.zeros((DIM, DIM)); P[np.arange(DIM), p] = 1; RING.append(P)
def rot(phi, th, om):
    c, s, a, b = np.cos(th / 2), np.sin(th / 2), (phi + om) / 2, (phi - om) / 2
    return np.array([[np.exp(-1j * a) * c, -np.exp(1j * b) * s], [np.exp(-1j * b) * s, np.exp(1j * a) * c]])
def layer_unitaries(w):
    out = []
    for l in range(w.shape[0]):
        K = np.array([[1.0 + 0j]])
        for i in range(N): K = np.kron(K, rot(*w[l, i]))
        out.append(K)
    return out
def encode(C):
    t = C * np.pi / 2
    psi = np.ones((len(C), 1), complex)
    for i in range(N):
        q = np.stack([np.cos(t[:, i] / 2), np.sin(t[:, i] / 2)], 1)
        psi = np.einsum('ba,bc->bac', psi, q).reshape(len(C), -1)
    return psi
def states(C, w):                        # pure state before measurement, (n, 16)
    psi = encode(C)
    for K in layer_unitaries(w):
        psi = psi @ K.T
        for P in RING: psi = psi @ P.T
    return psi
def expvals_pure(C, w):
    pr = np.abs(states(C, w)) ** 2
    return np.stack([pr @ z for z in ZOPS], 1)
def depol(rho, j, p):
    if p <= 0: return rho
    s = sum(P @ rho @ P.conj().T for P in PAULI[j])
    return (1 - 3 * p / 4) * rho + (p / 4) * s
def expvals_noisy(C, w, p1, p2, ro):
    psi = encode(C); rho = np.einsum('bi,bj->bij', psi, psi.conj())
    pairs = [(i, i + 1) for i in range(N - 1)] + [(N - 1, 0)]
    for K in layer_unitaries(w):
        rho = K @ rho @ K.conj().T
        for j in range(N): rho = depol(rho, j, p1)
        for P, (c, t) in zip(RING, pairs):
            rho = P @ rho @ P.T
            rho = depol(depol(rho, c, p2), t, p2)
    diag = np.real(np.einsum('bii->bi', rho))
    return (1 - 2 * ro) * np.stack([diag @ z for z in ZOPS], 1)
def meyer_wallach(psi):
    t = psi.reshape((-1,) + (2,) * N); purities = []
    for j in range(N):
        a = np.moveaxis(t, j + 1, 1).reshape(len(psi), 2, -1)
        r = np.einsum('bik,bjk->bij', a, a.conj()); purities.append(np.real(np.einsum('bij,bji->b', r, r)))
    return 2 * (1 - np.mean(purities, 0))
rng = np.random.default_rng(0); Ct = rng.uniform(-1, 1, (64, 4)); Wt = rng.uniform(0, 2 * np.pi, (2, 4, 3))
ref = circuit_expvals(Ct, Wt, NumpyOps, 4)
e1 = np.abs(expvals_pure(Ct, Wt) - ref).max(); e2 = np.abs(expvals_noisy(Ct, Wt, 0, 0, 0) - ref).max()
print('self-test vs training simulator: pure', e1, '| density matrix', e2); assert e1 < 1e-6 and e2 < 1e-6

In [ ]:
# 2 · Helpers: models, codes, function-class fits
def build(name, seed=None):
    tf.keras.backend.clear_session(); tf.keras.mixed_precision.set_global_policy('float32')
    if seed is not None: qseg2.set_seeds(seed)
    return qseg2.build_p2_model(name, SHAPE, cfg.n_qubits, cfg.n_q_layers)
def load(d, name):
    m = build(name); m.load_weights(f'{d}/best.weights.h5'); return m
def codes_in(m, X):                      # codes entering the circuit (output of pre_dense / pre_conv)
    cm = tf.keras.Model(m.input, m.get_layer('quantum_circuit').input)
    c = np.concatenate([cm(tf.constant(X[i:i + B]), training=False).numpy() for i in range(0, len(X), B)]).astype(np.float64)
    return c
def basis(C, maxorder):
    a = C * np.pi / 2; f = [np.ones_like(a), np.cos(a), np.sin(a)]; cols = []
    for k in itertools.product(range(3), repeat=N):
        if sum(1 for x in k if x) <= maxorder: cols.append(np.prod([f[k[j]][:, j] for j in range(N)], 0))
    return np.stack(cols, 1)
def r2_lin(Y, F):
    coef = np.linalg.lstsq(F, Y, rcond=None)[0]; sse = ((Y - F @ coef) ** 2).sum(); sst = ((Y - Y.mean(0)) ** 2).sum()
    return 1 - sse / sst
def r2_fit(Y, C, kind, restarts=5, seed=0):
    rng = np.random.default_rng(seed); a = Y.shape[0]; sst = ((Y - Y.mean(0)) ** 2).sum(); best = -np.inf
    def f(p, C):
        if kind == 'tr': A, ph = p[:16].reshape(4, 4), p[16:]; return np.tanh(np.cos(C * np.pi / 2 + ph) @ A.T)
        W, b = p[:16].reshape(4, 4), p[16:]; return np.tanh(C @ W.T + b)
    for r in range(restarts):
        p0 = np.concatenate([rng.normal(0, 1, 16), rng.uniform(0, 2 * np.pi, 4) if kind == 'tr' else rng.normal(0, 0.1, 4)])
        res = least_squares(lambda p: (f(p, C) - Y).ravel(), p0, max_nfev=4000)
        best = max(best, 1 - (res.fun ** 2).sum() / sst)
    return best
def subsample(C, n=2000, seed=0):
    return C if len(C) <= n else C[np.random.default_rng(seed).choice(len(C), n, replace=False)]

In [ ]:
# 3 · A: what each trained circuit computes, on the codes it receives (all 400 CBIS-DDSM crops); resumable
dst = f'{OUT}/A_circuit_function.csv'
done = pd.read_csv(dst) if os.path.exists(dst) else pd.DataFrame()
rows = done.to_dict('records')
for key, d in CIRC:
    s = json.load(open(f'{d}/summary.json')); rid = os.path.basename(d)
    if len(done) and ((done.run == rid) & (done.schedule == SCHED[key])).any(): continue
    m = load(d, s['model']); C = codes_in(m, XC).reshape(-1, 4); del m
    C = subsample(C)
    w_tr = load(d, s['model']).get_layer('quantum_circuit').get_weights()[0].astype(np.float64)
    w_in = build(s['model'], int(s['seed'])).get_layer('quantum_circuit').get_weights()[0].astype(np.float64)
    for state, w in (('init', w_in), ('trained', w_tr)):
        Y = expvals_pure(C, w)
        rows.append(dict(run=rid, schedule=SCHED[key], model=s['model'], fold=int(s['fold']), seed=int(s['seed']), state=state,
                         n_codes=len(C), code_sd=float(C.std(0).mean()), out_sd=float(Y.std(0).mean()),
                         r2_full81=r2_lin(Y, basis(C, 4)), r2_order1=r2_lin(Y, basis(C, 1)), r2_order2=r2_lin(Y, basis(C, 2)),
                         r2_trig_fit=r2_fit(Y, C, 'tr'), r2_dense_fit=r2_fit(Y, C, 'cl'),
                         meyer_wallach=float(meyer_wallach(states(C, w)).mean())))
    pd.DataFrame(rows).to_csv(dst, index=False); gc.collect()
    print(SCHED[key], rid, '| trained: order-1 R2 %.3f, trig-fit R2 %.3f, MW %.3f' % (rows[-1]['r2_order1'], rows[-1]['r2_trig_fit'], rows[-1]['meyer_wallach']), flush=True)
A = pd.DataFrame(rows)
print(A.groupby(['schedule', 'model', 'state'])[['r2_order1', 'r2_order2', 'r2_trig_fit', 'r2_dense_fit', 'meyer_wallach', 'out_sd']].median().round(3).to_string())

In [ ]:
# 4 · B: noisy re-evaluation of every trained circuit model (no retraining); resumable
NOISE = {'exact_np': (0, 0, 0), 'low': (0.0005, 0.005, 0.01), 'high': (0.002, 0.02, 0.03)}   # (p1 after Rot, p2 after CNOT, readout)
REPL_G = tf.Variable(tf.zeros((B, 4)), trainable=False); REPL_P = tf.Variable(tf.zeros((B, 4, 4, 4)), trainable=False)
USE = tf.Variable(0.0, trainable=False)
if not getattr(qseg2.Sever, '_addl', False):
    _sever_orig = qseg2.Sever.call
    def _sever_call(self, x):
        y = _sever_orig(self, x); r = REPL_G if len(x.shape) == 2 else REPL_P
        return tf.where(USE > 0.5, tf.cast(r[:tf.shape(x)[0]], y.dtype), y)
    qseg2.Sever.call = _sever_call; qseg2.Sever._addl = True
def forward(m):
    f = tf.function(lambda x: m(x, training=False), reduce_retracing=True)
    def run(X, repl=None):
        out = []
        for i in range(0, len(X), B):
            xb = X[i:i + B]
            if repl is None: USE.assign(0.0)
            else:
                r = np.zeros((B,) + repl.shape[1:], np.float32); r[:len(xb)] = repl[i:i + B]
                (REPL_G if repl.ndim == 2 else REPL_P).assign(r); USE.assign(1.0)
            out.append(f(tf.constant(xb)).numpy())
        USE.assign(0.0); return np.concatenate(out)
    return run
os.makedirs(f'{OUT}/B_noise', exist_ok=True)
for key, d in CIRC:
    s = json.load(open(f'{d}/summary.json')); rid = os.path.basename(d); f_out = f'{OUT}/B_noise/{SCHED[key]}_{rid}.csv'
    if os.path.exists(f_out): continue
    m = load(d, s['model']); run = forward(m); w = m.get_layer('quantum_circuit').get_weights()[0].astype(np.float64)
    _, _, te = qseg2.fold_split(DFC, int(s['fold'])); parts = []
    for ds, Xs, Ys, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, MD.roi_id.values)):
        Cin = codes_in(m, Xs); shp = Cin.shape; t = pd.DataFrame({'roi_id': ids})
        t['dice'] = qseg2.per_image_scores(Ys, run(Xs), cfg.threshold).dice.values
        for nm, (p1, p2, ro) in NOISE.items():
            Yn = expvals_noisy(Cin.reshape(-1, 4), w, p1, p2, ro).reshape(shp).astype(np.float32)
            t[f'dice_{nm}'] = qseg2.per_image_scores(Ys, run(Xs, Yn), cfg.threshold).dice.values
        t.insert(0, 'dataset', ds); parts.append(t)
    out = pd.concat(parts, ignore_index=True)
    out['schedule'], out['model'], out['fold'], out['seed'] = SCHED[key], s['model'], int(s['fold']), int(s['seed'])
    out.to_csv(f_out, index=False); del m; gc.collect()
    print(SCHED[key], rid, '| max |exact_np - exact|', float((out.dice_exact_np - out.dice).abs().max()),
          '| mean dDice low %.4f high %.4f' % ((out.dice_low - out.dice).mean(), (out.dice_high - out.dice).mean()), flush=True)
Bn = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/B_noise/*.csv'))], ignore_index=True)
Bn.to_csv(f'{OUT}/B_noise_per_image.csv.gz', index=False, compression='gzip', float_format='%.17g')
print(Bn.groupby(['dataset', 'schedule', 'model'])[['dice', 'dice_exact_np', 'dice_low', 'dice_high']].mean().round(4).to_string())

In [ ]:
# 5 · C: training dynamics and seed variability of all runs
rows = []
for key, d in ALL:
    s = json.load(open(f'{d}/summary.json'))
    rows.append(dict(run=os.path.basename(d), schedule=SCHED[key], model=s['model'], fold=int(s['fold']), seed=int(s['seed']),
                     best_epoch=s.get('best_epoch'), epochs_run=s.get('epochs_run'), dice_test_fold=s.get('dice_per_image')))
Cd = pd.DataFrame(rows); Cd.to_csv(f'{OUT}/C_training_dynamics.csv', index=False)
dyn = Cd.groupby(['schedule', 'model']).agg(runs=('run', 'size'), best_epoch_median=('best_epoch', 'median'),
                                             best_epoch_iqr=('best_epoch', lambda v: float(v.quantile(.75) - v.quantile(.25))),
                                             epochs_run_median=('epochs_run', 'median'))
sd = Cd.groupby(['schedule', 'model', 'fold']).dice_test_fold.std().groupby(['schedule', 'model']).mean().rename('seed_sd_mean_over_folds')
S = dyn.join(sd); S.to_csv(f'{OUT}/C_summary.csv'); print(S.round(4).to_string())
print('DONE')